# Playwright — Connect to existing Chrome (CDP)

Connects to your **running Chrome** browser via CDP, so the website sees your real profile, cookies, and session.

## Setup: start Chrome with remote debugging

Before running this notebook, start Chrome from the terminal:

```bash
/Applications/Google\ Chrome.app/Contents/MacOS/Google\ Chrome --remote-debugging-port=9222 --user-data-dir=/tmp/chrome-debug-profile
```

Once Chrome is open, run the cells below.

In [4]:
from playwright.async_api import async_playwright

In [5]:
async def attach_to_chrome():
    p = await async_playwright().start()
    browser = await p.chromium.connect_over_cdp("http://localhost:9222")
    context = browser.contexts[0]
    page = await context.new_page()
    print("Attached to Chrome via CDP")
    return p, browser, context, page


_p, _browser, _context, _page = await attach_to_chrome()

Attached to Chrome via CDP


### Navigate, scrape, interact

`_page` is a tab in your real Chrome. Use it normally.

In [7]:
await _page.goto("https://gallery.fotostudio.io/entre-nous-photographie/seance-grossesse-caroline-6?accessCode=wiLD5n")

<Response url='https://gallery.fotostudio.io/entre-nous-photographie/seance-grossesse-caroline-6?accessCode=wiLD5n' request=<Request url='https://gallery.fotostudio.io/entre-nous-photographie/seance-grossesse-caroline-6?accessCode=wiLD5n' method='GET'>>

In [15]:
await _page.locator("button.Cover__ScrollDown.Cover__ScrollDown--zeta").all()

[<Locator frame=<Frame name= url='https://gallery.fotostudio.io/entre-nous-photographie/seance-grossesse-caroline-6'> selector='button.Cover__ScrollDown.Cover__ScrollDown--zeta >> nth=0'>]

In [16]:
await _page.locator("button.Cover__ScrollDown.Cover__ScrollDown--zeta").click()

In [27]:
await (await _page.locator("img.pswp__img[src]").all())[0].get_attribute("src")

'https://d33i2hvz3orjoz.cloudfront.net/747/galleries/355425/w6am4UORSageH3MMg9/wm-web/1783514356_001B-2026-exemples-retouchees.jpg?response-content-disposition=attachment%3B%20filename%3D%221783514356_001B-2026-exemples-retouchees.jpg%22&Expires=1783611799&Key-Pair-Id=K39NNIRCH8GNM0&Signature=hu0FJa0hI~sFDSmIpgmXotFh8h2WtIAtPB5QXd2qlEWYs6MGpqhqaGXREuY0WhGPx-oEO108kfLHaHpEygu5Yazu~BZWLgrLzyO-F1noZdslTVL1SPEMDnx45Q-wW2Kbfiud16WLSedjgCW0H-tI6DP6pi5my~EpXKznL7D3j~KvKmrWbItszO6VF949t~1O-0bWYrHEPnt3tPveH6jD8bm7A0WLulb3pdo42XQM7RcQUq2SW-428EmrWZCFC3iPo6Z4bXQYDeo1jYyF4xjzEgwbsjeZGLzle7UL8ZYbI4teRnZ3nMFMRY1Y3BfZYQJh2IR1E0H0lYpFtCW1eZcCQA__'

In [51]:
img_url_selector = await _page.locator('div.pswp__item[aria-hidden="false"] img.pswp__img[src]').all()
img_name_selector = await _page.locator("div.pswp__photo-title.pswp__hide-on-close").all()
for img in img_url_selector:
    img_url = await img.get_attribute("src")
    print(img_url)
await img_name_selector[0].text_content()

https://d33i2hvz3orjoz.cloudfront.net/747/galleries/355425/w6am4UORSageH3MMg9/wm-web/1783514356_001B-2026-exemples-retouchees.jpg?response-content-disposition=attachment%3B%20filename%3D%221783514356_001B-2026-exemples-retouchees.jpg%22&Expires=1783611799&Key-Pair-Id=K39NNIRCH8GNM0&Signature=hu0FJa0hI~sFDSmIpgmXotFh8h2WtIAtPB5QXd2qlEWYs6MGpqhqaGXREuY0WhGPx-oEO108kfLHaHpEygu5Yazu~BZWLgrLzyO-F1noZdslTVL1SPEMDnx45Q-wW2Kbfiud16WLSedjgCW0H-tI6DP6pi5my~EpXKznL7D3j~KvKmrWbItszO6VF949t~1O-0bWYrHEPnt3tPveH6jD8bm7A0WLulb3pdo42XQM7RcQUq2SW-428EmrWZCFC3iPo6Z4bXQYDeo1jYyF4xjzEgwbsjeZGLzle7UL8ZYbI4teRnZ3nMFMRY1Y3BfZYQJh2IR1E0H0lYpFtCW1eZcCQA__


'001B-2026-exemples-retouchees.jpg'

In [ ]:
await _page.locator("div.pswp__photo-title.pswp__hide-on-close").all()

[<Locator frame=<Frame name= url='https://gallery.fotostudio.io/entre-nous-photographie/seance-grossesse-caroline-6'> selector='div.pswp__photo-title.pswp__hide-on-close >> nth=0'>]

In [ ]:
# pswp__photo-title pswp__hide-on-close

In [53]:
import time

import requests

img_url_set = set()

In [59]:
while True:
    time.sleep(0.2)
    img_selector = await _page.locator("img.pswp__img[src]").all()
    for img in img_selector:
        img_url = await img.get_attribute("src")
        img_url_selector = await _page.locator('div.pswp__item[aria-hidden="false"] img.pswp__img[src]').all()
        img_name_selector = await _page.locator("div.pswp__photo-title.pswp__hide-on-close").all()

        if len(img_url_selector) > 1:
            raise (ValueError(f"length of selector {len(img_url_selector)}"))
        if len(img_name_selector) > 1:
            raise (ValueError(f"length of selector {len(img_name_selector)}"))

        img_url = await img_url_selector[0].get_attribute("src")
        img_name = await img_name_selector[0].text_content()

        if img_url not in img_url_set:
            img_response = requests.get(img_url)
            with open(f"images_by_name/{img_name}", "wb") as f:
                f.write(img_response.content)
                print(f"saved {img_name}")
            img_url_set.add(img_url)

saved 20260601-G37A2833-251.jpg


CancelledError: 

In [ ]:
import requests

In [29]:
response = requests.get(
    "https://d33i2hvz3orjoz.cloudfront.net/747/galleries/355425/w6am4UORSageH3MMg9/wm-web/1783514356_001B-2026-exemples-retouchees.jpg?response-content-disposition=attachment%3B%20filename%3D%221783514356_001B-2026-exemples-retouchees.jpg%22&Expires=1783611799&Key-Pair-Id=K39NNIRCH8GNM0&Signature=hu0FJa0hI~sFDSmIpgmXotFh8h2WtIAtPB5QXd2qlEWYs6MGpqhqaGXREuY0WhGPx-oEO108kfLHaHpEygu5Yazu~BZWLgrLzyO-F1noZdslTVL1SPEMDnx45Q-wW2Kbfiud16WLSedjgCW0H-tI6DP6pi5my~EpXKznL7D3j~KvKmrWbItszO6VF949t~1O-0bWYrHEPnt3tPveH6jD8bm7A0WLulb3pdo42XQM7RcQUq2SW-428EmrWZCFC3iPo6Z4bXQYDeo1jYyF4xjzEgwbsjeZGLzle7UL8ZYbI4teRnZ3nMFMRY1Y3BfZYQJh2IR1E0H0lYpFtCW1eZcCQA__"
)

In [32]:
with open("image1.jpg", "wb") as f:
    f.write(response.content)

In [6]:
await _page.goto("https://quotes.toscrape.com/js/")
await _page.wait_for_selector(".quote")

# quotes = []
# while True:
#     page_quotes = await _page.locator(".quote").all()
#     for q in page_quotes:
#         quotes.append(await q.text_content())

#     next_button = await _page.locator(".next a").all()
#     if next_button:
#         await next_button[0].click()
#         await _page.wait_for_load_state("domcontentloaded")
#     else:
#         break

# len(quotes)

<JSHandle preview=JSHandle@node>

In [ ]:
# Open another tab in the same session
page2 = await _context.new_page()
await page2.goto("https://quotes.toscrape.com/login")
print(f"Tab 1: {_page.url}")
print(f"Tab 2: {page2.url}")

### Cleanup

Close pages opened by this notebook (the browser stays open since Playwright doesn't own it):

In [ ]:
if _page and not _page.is_closed():
    await _page.close()
if "page2" in dir() and not page2.is_closed():
    await page2.close()
print("Pages closed.")